# 01_play_history

DDL: `bronze_play_history` — Raw listening history from `/me/player/recently-played`.
Each row is one play event as returned by Spotify, plus ingestion metadata.

In [ ]:
%run ../../tools/config/settings

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}.bronze_play_history (
    played_at           TIMESTAMP   COMMENT 'UTC timestamp of the play event',
    track_id            STRING      COMMENT 'Spotify track ID',
    track_name          STRING,
    track_duration_ms   BIGINT,
    artist_ids          ARRAY<STRING>,
    artist_names        ARRAY<STRING>,
    album_id            STRING,
    album_name          STRING,
    context_type        STRING      COMMENT 'playlist / artist / album / null',
    context_href        STRING,
    _raw                STRING      COMMENT 'Full serialized API item JSON',
    run_id              STRING      COMMENT 'Pipeline run identifier',
    ingestion_date      DATE        COMMENT 'Partition key'
)
USING DELTA
PARTITIONED BY (ingestion_date)
COMMENT 'Bronze: raw listening history from Spotify /me/player/recently-played'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true',
    'delta.columnMapping.mode'         = 'name'
)
""")

print(f"Table {CATALOG}.{BRONZE_SCHEMA}.bronze_play_history ready.")